<p align="center">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>


# Substantia Nigra Cell Subtyping 
## Case Study — Part 03: MapMyCells

**ASAP CRN Learning Lab**  
Reproducible exploratory and meta-analysis examples using ASAP CRN data

---

### Overview

This notebook continues the Substantia Nigra case study by applying **MapMyCells** to align Substantia Nigra–derived single cells to an external Basal Ganglia reference taxonomy ([Allen Institute HMBA-BG](https://alleninstitute.github.io/abc_atlas_access/descriptions/HMBA-BG_dataset.html). 

Rather than performing explicit batch integration, this approach focuses on direct cell-type mapping, enabling rapid biological interpretation and comparison to established cell atlases.

---

### Learning Objectives

By the end of this notebook, you will be able to:

- run reference-based cell type mapping with MapMyCells
- interpret key output fields such as `class_name`, `rho`, and `prob`
- compare reference-based labels with marker-based validation
- apply simple rules to refine low-confidence annotations
- export annotated data for downstream analysis
---


### Prerequisites

- Completion of Case Study — Part 02: Preprocessing and Feature Selection
- Access to the analysis-ready Substantia Nigra AnnData artifact

---

### Inputs

- **Analysis-ready Substantia Nigra AnnData object**
    - Example: `asap-{dataset_team}__sn_cells__preprocessed.h5ad`
---

### Outputs

This notebook generates the following AnnData artifacts:

- **Taxonomy-mapped Substantia Nigra AnnData object**
    - Example: `sn_mapmycells_integrated_processed_output.h5ad`
    - Contains:
        - Substantia Nigra–derived cells
        - Reference-aligned cell-type annotations stored in .obs
        - Mapping confidence scores and metadata
        - Normalized and log-transformed expression values
        - Preprocessing outputs (e.g., PCA stored in .obsm)ebooks  
- **Psuedobulk AnnData objects**
    - Pseudobulk aggregates by refined cell type, split by condition

---

> **Notes**
> Reference taxonomy: Basal Ganglia (Allen Institute HMBA-BG)
https://alleninstitute.github.io/abc_atlas_access/descriptions/HMBA-BG_dataset.html

## Table of Contents

1. [Package Imports and Configuration](#1-package-imports-and-configuration)
2. [Data Sources and Context](#2-data-sources-and-context)
3. [Reference-Based Mapping with MapMyCells with MapMyCells](#3-reference-based-mapping)
4. [Analyze and Integrate MapMyCells Results](#4-analyze-and-integrate-mapmycells-results)
5. [Prepare Data for Downstream Analysis](#5-prepare-data)
6. [Dopaminergic (DA) Marker Analysis and Scoring](#6-dopaminergic-da-marker-analysis-and-scoring)
7. [Export Data](#7-export-data)

## 1. Package Imports and Configuration

In [ ]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import os
from pathlib import Path

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime


## 2. Data sources and Context

### 2.1 Set dataset paths
In this example, we are working with the **PMDBS single‑cell RNA‑seq cohort** dataset:

- **Workflow** → `pmdbs_sc_rnaseq`  
- **Team** → `cohort`  
- **Source** → `pmdbs`  
- **Type** → `sc-rnaseq`  

These components are combined to construct the bucket and dataset names.  
We then set the path to the **cohort analysis outputs** and preview the available files.


In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

In [ ]:
# Build and set path to desired dataset
DATASETS_PATH = WS_ROOT / "01_PMDBS"

workflow       = "pmdbs_sc_rnaseq"   # FIXED: lowercase
dataset_team   = "cohort"
dataset_source = "pmdbs"
dataset_type   = "sc-rnaseq"

bucket_name  = f"asap-curated-{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / "PMDBS_sc_rnaseq" / bucket_name / workflow
print("Dataset Path:", dataset_path)

cohort_analysis_path = dataset_path / "cohort_analysis"

# Define the local path for case study output
local_data_path = WS_FILES / "sn_celltyping"
!ls {local_data_path}

# Define output paths
plots_path         = local_data_path / "output_plots"
output_path        = local_data_path / "output_matrices"
mapmycells_input_dir  = local_data_path / "mapmycells" / "input"
mapmycells_output_dir = local_data_path / "mapmycells" / "output"

# Ensure all directories exist
for d in [plots_path, output_path, mapmycells_input_dir, mapmycells_output_dir]:
    os.makedirs(d, exist_ok=True)

## 3. Reference-Based Mapping with MapMyCells

[MapMyCells](https://portal.brain-map.org/atlases-and-data/bkp/mapmycells) performs **direct taxonomy mapping** of Substantia Nigra–derived cells to a Basal Ganglia reference. Rather than integrating datasets (as in batch-correction approaches like scVI or Harmony), it uses **marker-gene correlation** to assign each cell a taxonomy label and a confidence score.


### 3.1 Set Up MapMyCells Dependencies

We import the `cell_type_mapper` package (which implements the MapMyCells algorithm) and the `abc_atlas_access` package (which provides utilities for downloading reference taxonomy files from the Allen Brain Atlas).

In [ ]:
import json
import cell_type_mapper
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from cell_type_mapper.cli.from_specified_markers import FromSpecifiedMarkersRunner

### 3.2 Download Reference Taxonomy Files

MapMyCells requires two precomputed reference files derived from the Basal Ganglia taxonomy:

1. **Precomputed statistics** (`.h5`) — reference cluster expression profiles used for correlation-based assignment  
2. **Query markers** (`.json`) — the set of marker genes used to compare query cells against the reference  

These files are downloaded once and reused across mapping runs.

In [ ]:
# Define paths for MapMyCells reference inputs
precomputed_stats_filepath = (
    mapmycells_input_dir / "Human.precomputed_stats.20250507.h5"
)
query_markers_filepath = (
    mapmycells_input_dir / "Human.query_markers.20250507.json"
)

# Download Basal Ganglia reference taxonomy files (Allen Institute HMBA-BG)
! wget "https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com/HMBA/BasalGanglia/BICAN_05072025_pre-print_release/MapMyCells/Human.precomputed_stats.20250507.h5" \
    -O {precomputed_stats_filepath}

! wget "https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com/HMBA/BasalGanglia/BICAN_05072025_pre-print_release/MapMyCells/Human.query_markers.20250507.json" \
    -O {query_markers_filepath}


### 3.3 Configure MapMyCells

Before running MapMyCells, we configure its settings. Key parameters include:

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `normalization` | `"raw"` | Tells MapMyCells our input contains raw counts (normalization is handled internally) |
| `n_processors` | `4` | Number of parallel workers |
| `bootstrap_factor` | `0.5` | Fraction of marker genes resampled per bootstrap iteration |
| `bootstrap_iteration` | `100` | Number of bootstrap rounds to estimate assignment stability |

We also limit thread counts for numerical libraries to avoid oversubscription on shared compute environments.

In [ ]:
import os
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"


# paths to files where mapping output will be written
json_dst_path = str( mapmycells_output_dir / "human_sn_mapping.json") 
csv_dst_path = str( mapmycells_output_dir / "human_sn_mapping.csv")

sn_processed_filename = (
    local_data_path / f"asap-{dataset_team}_sn_cells__preprocessed.h5ad"
)

config = {
    "query_path": str(sn_processed_filename),
    "extended_result_path": json_dst_path,
    "csv_result_path": csv_dst_path,
    "verbose_csv": True,
    "query_markers": {
       "serialized_lookup":str(query_markers_filepath)
    },
    "precomputed_stats": {
        "path": str(precomputed_stats_filepath)
    },
    "type_assignment": {
        "n_processors": 4,
        "normalization": "raw",
        "bootstrap_factor": 0.5,
        "bootstrap_iteration": 100
    }
}

### 3.4 Run MapMyCells

In [ ]:
# run map my cells
runner = FromSpecifiedMarkersRunner(
    args=[],
    input_data=config
)
runner.run()

> ⏱️ **Expected runtime:** ~30 minutes depending on dataset size and available compute when run for the first time.

## 4. Analyze and Integrate MapMyCells Results

In this section, we load, standardize, and evaluate the MapMyCells (MMC) outputs.

The goal is to transform raw MMC outputs into:
- **Interpretable cell type annotations** — simplified labels like "Dopaminergic", "GABAergic", etc.  
- **Consistent fields** — standardized column names across different reference workflows  
- **Analysis-ready data** — merged into the AnnData object for visualization and downstream use  

This step bridges **raw mapping output → usable biological annotations**.

### 4.1 Load MapMyCells Results

The MapMyCells CSV output contains one row per cell with:
- **Predicted labels** at each taxonomy level (`class_name`, `subclass_name`, `supertype_name`)  
- **Correlation coefficients** — how similar the cell's expression is to the assigned reference profile  
- **Bootstrapping probabilities** — how consistently the cell maps to the same label across resampled gene sets  

In [ ]:
mmc_output = pd.read_csv(csv_dst_path, comment='#')
mmc_output.columns

### 4.2 Reformat MapMyCells Results
The raw MMC outputs contain detailed but workflow-specific fields (e.g., `class_name`, `subclass_name`, correlation metrics).

We reformat these results to:

- create consistent, intuitive fields across workflows
- derive a simplified cell_type label
- extract key metrics for interpretation:
    - `rho` (correlation strength)
    - `prob` (bootstrapping confidence)

This function performs four tasks:

1. groups detailed MapMyCells labels into broader categories
2. selects the relevant confidence metrics
3. assigns a simplified `cell_type`
4. flags low-confidence assignments as `Unknown`

In [ ]:
def summarize_mmc_results(mmc_results: pd.DataFrame, workflow_name: str):
    """
    Summarize MMC results for human, mouse, or basal ganglia workflows.

    Parameters
    ----------
    mmc_results : pd.DataFrame
        Input dataframe containing MMC outputs.
    workflow_name : str
        One of: "pmdbs_sc_rnaseq", "mouse_sc_rnaseq", "basal_ganglia".

    Returns
    -------
    pd.DataFrame
        Summary dataframe with standardized fields.
    """

    # ------------------------------------------------------------
    # pmdbs_sc_rnaseq
    # ------------------------------------------------------------
    if workflow_name == "pmdbs_sc_rnaseq":
        gabaergic = mmc_results["class_name"] == "Neuronal: GABAergic"
        glutamatergic = mmc_results["class_name"] == "Neuronal: Glutamatergic"
        non_neuronal = mmc_results["class_name"] == "Non-neuronal and Non-neural"

        mmc_results.loc[gabaergic, "phenotype"] = "GABAergic"
        mmc_results.loc[glutamatergic, "phenotype"] = "Glutamatergic"
        mmc_results.loc[non_neuronal, "phenotype"] = mmc_results.loc[
            non_neuronal, "subclass_name"
        ]

        mmc_results.loc[glutamatergic, "rho"] = mmc_results.loc[
            glutamatergic, "class_correlation_coefficient"
        ]
        mmc_results.loc[gabaergic, "rho"] = mmc_results.loc[
            gabaergic, "class_correlation_coefficient"
        ]
        mmc_results.loc[non_neuronal, "rho"] = mmc_results.loc[
            non_neuronal, "subclass_correlation_coefficient"
        ]

        mmc_results.loc[glutamatergic, "prob"] = mmc_results.loc[
            glutamatergic, "class_bootstrapping_probability"
        ]
        mmc_results.loc[gabaergic, "prob"] = mmc_results.loc[
            gabaergic, "class_bootstrapping_probability"
        ]
        mmc_results.loc[non_neuronal, "prob"] = mmc_results.loc[
            non_neuronal, "subclass_bootstrapping_probability"
        ]

        mmc_results["cell_type"] = mmc_results["phenotype"]

        # Confidence filtering
        mmc_results.loc[mmc_results["rho"] < 0.5, "cell_type"] = "Unknown"
        mmc_results.loc[mmc_results["prob"] < 0.5, "cell_type"] = "Unknown"

        return mmc_results[
            [
                "cell_id"
                "cell_type",
                "phenotype",
                "rho",
                "prob",
                "class_name",
                "subclass_name",
                "supertype_name",
            ]
        ]

    # ------------------------------------------------------------
    # mouse_sc_rnase
    # ------------------------------------------------------------
    elif workflow_name == "mouse_sc_rnaseq":
        gabaergic = mmc_results["class_name"].str.contains("GABA")
        glutamatergic = mmc_results["class_name"].str.contains("Glut")
        serotonergic = mmc_results["class_name"].str.contains("Sero")
        non_neuronal = ~(gabaergic | glutamatergic | serotonergic)

        mmc_results.loc[gabaergic, "phenotype"] = "GABAergic"
        mmc_results.loc[glutamatergic, "phenotype"] = "Glutamatergic"
        mmc_results.loc[serotonergic, "phenotype"] = "Serotonergic"
        mmc_results.loc[non_neuronal, "phenotype"] = mmc_results.loc[
            non_neuronal, "subclass_name"
        ]

        mmc_results.loc[glutamatergic, "rho"] = mmc_results.loc[
            glutamatergic, "class_correlation_coefficient"
        ]
        mmc_results.loc[gabaergic, "rho"] = mmc_results.loc[
            gabaergic, "class_correlation_coefficient"
        ]
        mmc_results.loc[serotonergic, "rho"] = mmc_results.loc[
            serotonergic, "class_correlation_coefficient"
        ]
        mmc_results.loc[non_neuronal, "rho"] = mmc_results.loc[
            non_neuronal, "subclass_correlation_coefficient"
        ]

        mmc_results.loc[glutamatergic, "prob"] = mmc_results.loc[
            glutamatergic, "class_bootstrapping_probability"
        ]
        mmc_results.loc[gabaergic, "prob"] = mmc_results.loc[
            gabaergic, "class_bootstrapping_probability"
        ]
        mmc_results.loc[serotonergic, "prob"] = mmc_results.loc[
            serotonergic, "class_bootstrapping_probability"
        ]
        mmc_results.loc[non_neuronal, "prob"] = mmc_results.loc[
            non_neuronal, "subclass_bootstrapping_probability"
        ]

        mmc_results["cell_type"] = mmc_results["phenotype"]

        mmc_results.loc[mmc_results["rho"] < 0.5, "cell_type"] = "Unknown"
        mmc_results.loc[mmc_results["prob"] < 0.5, "cell_type"] = "Unknown"

        return mmc_results[
            [
                "cell_type",
                "phenotype",
                "rho",
                "prob",
                "class_name",
                "subclass_name",
                "supertype_name",
                "cluster_name",
                "cluster_correlation_coefficient",
                "cluster_bootstrapping_probability",
            ]
        ]

    # ------------------------------------------------------------
    # BASAL GANGLIA
    # ------------------------------------------------------------
    elif workflow_name == "basal_ganglia":
        class_mapper = {
            "OPC-Oligo": "OPC-Oligo",
            "Astro-Epen": "Astrocyte",
            "Vascular": "Vascular",
            "Immune": "Immune",
            "F M Glut": "Glutamatergic",
            "M Dopa": "Dopaminergic",
            "CN CGE GABA": "GABAergic",
            "F M GABA": "GABAergic",
            "CN MGE GABA": "GABAergic",
            "CN LGE GABA": "GABAergic",
            "Cx GABA": "GABAergic",
            "CN GABA-Glut": "GABAergic",
        }

        mmc_results["phenotype"] = mmc_results["Class_name"].map(class_mapper)

        gabaergic = mmc_results["Class_name"] == "GABAergic"
        mmc_results["rho"] = mmc_results["Class_correlation_coefficient"]
        mmc_results.loc[gabaergic, "rho"] = mmc_results.loc[
            gabaergic, "Neighborhood_correlation_coefficient"
        ]

        mmc_results["prob"] = mmc_results["Class_bootstrapping_probability"]
        mmc_results.loc[gabaergic, "prob"] = mmc_results.loc[
            gabaergic, "Neighborhood_bootstrapping_probability"
        ]

        mmc_results["cell_type"] = mmc_results["phenotype"]
        mmc_results.loc[mmc_results["rho"] < 0.2, "cell_type"] = "Unknown"
        mmc_results.loc[mmc_results["prob"] < 0.5, "cell_type"] = "Unknown"

        name_mapper = {
            "Neighborhood_name": "supertype_name",
            "Class_name": "class_name",
            "Subclass_name": "subclass_name",
        }
        mmc_results.rename(columns=name_mapper, inplace=True)

        return mmc_results[
            [
                "cell_id",
                "cell_type",
                "phenotype",
                "rho",
                "prob",
                "class_name",
                "subclass_name",
                "supertype_name"
            ]
        ]

    # ------------------------------------------------------------
    # UNKNOWN WORKFLOW
    # ------------------------------------------------------------
    else:
        raise ValueError(
            f"[ERROR] Source cannot be detected from workflow name: [{workflow_name}]"
        )

In [ ]:
mmc_res = summarize_mmc_results(mmc_output, "basal_ganglia")
mmc_res.head()

### 4.4 Assess MMC Results

After reformatting, we evaluate the quality and distribution of the assignments.

Key aspects to assess:

- distribution of cell_type labels
- proportion of `"Unknown"` assignments (low-confidence cells)
- ranges and distributions of `rho` (correlation strength) and `prob` (assignment stability)

This helps answer:
- Are the assignments biologically reasonable for this dataset?
- Are confidence thresholds too strict or too lenient?
- Do certain cell populations show weaker mapping signals?

In [ ]:
# Set index to cell_id for easy alignment
adata = sc.read_h5ad(sn_processed_filename)
mapmycells_df = mmc_res.set_index("cell_id")

# validate connection 
print(len(set(mapmycells_df.index).intersection(set(adata.obs_names))))

# assign directly 
for col in ["cell_type", "class_name", "subclass_name", "supertype_name", "phenotype", "rho", "prob"]: 
    adata.obs[col] = mapmycells_df[col].values

In [ ]:
adata

#### Visualize class-level annotations on UMAP

A quick sanity check: plotting the assigned `class_name` and `prob` (bootstrapping confidence) on the UMAP embedding from Part 02. Cells should cluster coherently by class, and confidence should be higher in dense, well-separated regions.

In [ ]:
adata.obsm["X_umap"] = adata.obsm.pop("_X_umap")
sc.pl.umap(adata, color=["class_name", "prob"])

#### Restore gene names and save intermediate checkpoint

We set `var_names` to human-readable gene symbols for downstream plotting (e.g., dot plots, marker expression). We also save a checkpoint of the AnnData object with MapMyCells annotations before further processing.

In [ ]:
# move gene names back to index for readability
# 2. Set var_names to gene_names
adata.var_names = adata.var["gene_name"].astype(str)

# 3. Ensure uniqueness
adata.var_names_make_unique()

# Quick check
print(adata.var.head())
print(adata.var_names[:5])

mmc_adata_output_filepath = ( mapmycells_output_dir / "sn_mapmycells_integrated_output.h5ad")
adata.write_h5ad(mmc_adata_output_filepath)

## 5. Prepare Data for Downstream Analysis

With MapMyCells annotations in place, we now prepare the dataset for downstream analysis and visualization.

This step applies standard single-cell preprocessing:
- **Normalize** expression values across cells (library-size correction to 10,000 counts)
- **Log-transform** to stabilize variance  
- **Identify highly variable genes (HVGs)** for dimensionality reduction  
- **Cluster** cells using the Leiden algorithm  

> **Why preprocess again?** The AnnData from Part 02 contains raw counts and a pre-computed embedding. Here we normalize and log-transform for gene-level analyses (scoring, dot plots) and run clustering to explore within-annotation structure.

In [ ]:
# Normalize counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform the data
sc.pp.log1p(adata)

# Identify highly variable genes
hvgs_res =sc.experimental.pp.highly_variable_genes(
        adata,
        n_top_genes=3000,
        batch_key="sample", #using sample not batch
        flavor="pearson_residuals",
        check_values=True,
        layer="counts",
        subset=False,
        inplace=False
)

In [ ]:
# Leiden clustering at resolution 1.0
# This creates data-driven clusters independent of MapMyCells labels,
# which we can later compare against the reference-based annotations.
sc.tl.leiden(adata, resolution=1.0, key_added="leiden_sn_scVI_1.0", random_state=0)

## 6. Dopaminergic (DA) Marker Analysis and Scoring

To further validate and interpret MapMyCells annotations, we assess expression of canonical **dopaminergic (DA) marker genes**.

#### DA marker genes
- [TH](https://www.genecards.org/cgi-bin/carddisp.pl?gene=TH&keywords=TH)  
- [DDC](https://www.genecards.org/cgi-bin/carddisp.pl?gene=DDC&keywords=DDC)  
- [SLC6A3](https://www.genecards.org/cgi-bin/carddisp.pl?gene=SLC6A3&keywords=SLC6A3)  
- [NR4A2](https://www.genecards.org/cgi-bin/carddisp.pl?gene=NR4A2)  
- [SLC18A2](https://www.genecards.org/cgi-bin/carddisp.pl?gene=SLC18A2&keywords=SLC18A2)  

#### Biological rationale

These markers collectively capture key aspects of dopaminergic neuron identity:

1. **Dopamine synthesis** — *TH*, *DDC*: core enzymes required for dopamine production  
2. **Dopamine transport and storage** — *SLC6A3* (DAT), *SLC18A2* (VMAT2): regulate dopamine reuptake and vesicular packaging  
3. **Transcriptional identity** — *NR4A2* (Nurr1): a critical transcription factor for DA neuron development and survival  

#### Why this matters

MapMyCells provides reference-based cell type assignments. Marker-based scoring offers an **orthogonal validation layer** by directly assessing gene expression. This allows us to:
- Confirm dopaminergic identity in mapped cells  
- Identify cells with partial or ambiguous DA signatures  
- Compare marker-driven signals with reference-based annotations  
- Explore heterogeneity within DA populations  

#### References
- https://www.nature.com/articles/s41593-022-01061-1  
- https://www.cell.com/cell-reports/references/S2211-1247(14)00862-6  
- https://www.pnas.org/doi/10.1073/pnas.2410331121  
- https://pmc.ncbi.nlm.nih.gov/articles/PMC7285906/  
- https://elifesciences.org/articles/101035#s2  
- https://www.science.org/doi/10.1126/sciadv.adi8287

### 6.1 Compute DA Marker Score

In [ ]:
# Define DA marker genes and the cluster key for grouping
marker_genes = ["TH", "DDC", "SLC6A3", "NR4A2", "SLC18A2"]
cluster_key = "leiden_sn_scVI_1.0"

# Check for missing markers (gene names depend on var_names set above)
missing = [g for g in marker_genes if g not in adata.var_names]
if missing:
    warnings.warn(f"These marker genes are missing from adata.var_names and will be skipped: {missing}")
existing_markers = [g for g in marker_genes if g in adata.var_names]
if len(existing_markers) == 0:
    raise ValueError("No marker genes found in adata.var_names. Aborting.")

print(f"Using {len(existing_markers)}/{len(marker_genes)} markers: {existing_markers}")

In [ ]:
# Score DA neuron markers
# use_raw depends on whether you want to score on raw counts; set to True if adata.raw exists and is desired
sc.tl.score_genes(adata, gene_list=existing_markers, score_name="DA_score", use_raw=False)

In [ ]:
# Ensure cluster_key exists
if cluster_key not in adata.obs.columns:
    raise KeyError(f"Cluster key '{cluster_key}' not found in adata.obs")

### 6.2 Visualize DA Scores

**What to look for:**  
Dopaminergic marker scores should concentrate in a limited set of clusters rather than appearing uniformly across the embedding. Broad or diffuse signal may suggest weaker specificity or mixed populations.

In [ ]:
# UMAP plotting (assumes neighbors/pca/umap already computed; compute if needed)
sc.settings.figdir = str(plots_path)
try:
    sc.pl.umap(adata, color=[cluster_key, "DA_score"], cmap="viridis", save="_umap_DA_score.png", show=False)
except Exception:
    # fallback: compute UMAP if missing
    if "X_umap" not in adata.obsm_keys():
        sc.pp.neighbors(adata, use_rep="X_scVI" if "X_scVI" in adata.obsm_keys() else None)
        sc.tl.umap(adata)
    sc.pl.umap(adata, color=[cluster_key, "DA_score"], cmap="viridis", save="_umap_DA_score.png", show=True)


In [ ]:
# Dotplot for DA markers across clusters
sc.pl.dotplot(
    adata,
    var_names=existing_markers,
    groupby=cluster_key,
    standard_scale="var",
    title="Dopaminergic Markers",
    save="_DotPlot_DA-markers.png",
    show=True,
)

In [ ]:
# Group by cluster and compute mean DA_score
cluster_means = (
    adata.obs
    .groupby(adata.obs[cluster_key].astype(str), observed=False)["DA_score"]
    .mean()
    .reset_index()
    .rename(columns={cluster_key: "cluster"})
)

# Barplot of mean DA_score per cluster
plt.figure(figsize=(8, 4))
order = cluster_means.sort_values("DA_score", ascending=False)["cluster"].tolist()
sns.barplot(data=cluster_means, x="cluster", y="DA_score", palette="magma", order=order)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean DA marker score")
plt.xlabel("Cluster")
plt.title("Cluster enrichment for DA neuron markers")
plt.tight_layout()
plt.show()

In [ ]:
# Binary labels
adata.obs["is_DA"] = adata.obs["cell_type"] == "Dopaminergic"

# Violin by class
plt.figure(figsize=(6, 4))
order = adata.obs["cell_type"].value_counts().index.tolist() if "cell_type" in adata.obs.columns else None
sns.violinplot(x="cell_type", y="DA_score", data=adata.obs, order=order)
plt.xticks(rotation=45, ha="right")
plt.title("DA score by MapMyCells class")
plt.tight_layout()
plt.show()


### 6.3 Quantitative Concordance: DA Score vs. MapMyCells Labels

In [ ]:
# Pearson correlation between DA_score and MapMyCells confidence (prob)
corr = adata.obs[["DA_score", "prob"]].dropna().corr().iloc[0, 1]
print("Pearson corr (DA_score vs prob):", corr)

# ROC AUC: how well does DA_score separate DA from non-DA cells?
y_true = adata.obs["is_DA"].astype(int)
y_score = adata.obs["DA_score"].fillna(0).values

auc = roc_auc_score(y_true, y_score)
print("ROC AUC (DA_score -> MapMyCells DA):", auc)

# Precision / recall at the 75th percentile threshold
score_thresh = np.quantile(adata.obs["DA_score"].dropna(), 0.75)
y_pred = (adata.obs["DA_score"] >= score_thresh).astype(int)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
print(f"Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows=true 0/1, cols=pred 0/1):\n", cm)

#### How to interpret these metrics

| Metric | What it measures | Interpretation |
|--------|------------------|----------------|
| **Pearson correlation** | Agreement between `DA_score` and MapMyCells `prob` | Low correlation is expected — these capture different signals (marker expression vs. reference similarity) |
| **ROC AUC** | How well `DA_score` separates DA from non-DA cells across all thresholds | Values near 1.0 indicate strong discrimination |
| **Precision** | Fraction of predicted DA cells that are truly DA | High precision = few false positives |
| **Recall** | Fraction of true DA cells correctly identified | High recall = few missed DA cells |
| **F1 score** | Harmonic mean of precision and recall | Balances both error types |

High recall with lower precision suggests the score captures most DA cells but also flags some non-DA cells. This is expected when using a fixed percentile threshold rather than an optimized cutoff.

The **confusion matrix** provides the raw counts: true negatives, false positives, false negatives, and true positives.

> **Key takeaway:** These metrics help assess whether `DA_score` is better suited for **ranking/enrichment** (continuous use) or **binary classification** (threshold-based), and whether threshold adjustment may improve performance.

### 6.4 DA Labeling Refinement

We now refine the MapMyCells `"Dopaminergic"` label using a **multi-criteria filter**. A cell retains the DA label only if:

1. Its `DA_score` ≥ the 75th percentile threshold (high marker expression)  
2. Its `prob` ≥ 0.7 (high MapMyCells confidence)  

Cells labeled "Dopaminergic" by MapMyCells but failing either criterion are reclassified as `"Unknown"`. This combines reference-based and marker-based evidence for higher-confidence annotations.

> **Why these thresholds?**
> These cutoffs are intentionally conservative and are meant as an example of confidence-based refinement, not a universal rule. Depending on the dataset, users may wish to tune these thresholds based on marker distributions, class balance, or known biology.

In [ ]:
# Apply multi-criteria refinement to Dopaminergic labels
obs = adata.obs
prob_thresh = 0.7

# Validate required columns
assert "DA_score" in obs.columns, "DA_score missing"
assert "prob" in obs.columns, "prob missing"
assert "cell_type" in obs.columns, "cell_type missing"

# Create boolean masks
is_dopa    = obs["cell_type"] == "Dopaminergic"
high_score = obs["DA_score"] >= score_thresh
high_prob  = obs["prob"] >= prob_thresh

# Build refined labels: demote low-confidence DA cells to Unknown
refined = obs["cell_type"].copy()
refined.loc[is_dopa & ~(high_score & high_prob)] = "Unknown"
adata.obs["cell_type_refined"] = refined

In [ ]:
# Compare label counts before and after refinement
print("Before refinement")
print(obs["cell_type"].value_counts())
print("\nAfter refinement")
print(adata.obs["cell_type_refined"].value_counts())

In [ ]:
proportions = (
    adata.obs["cell_type_refined"]
    .value_counts(normalize=True)   # normalize=True → proportions
    .sort_values(ascending=False)
)

proportions


In [ ]:
proportions = (
    adata.obs["cell_type_refined"]
    .value_counts(normalize=True)
    .sort_values()
)

proportions.plot(kind="barh", figsize=(8,5))
plt.xlabel("Proportion of cells")
plt.title("Distribution of MapMyCells cell classes")
plt.show()


## 7. Export Data

We save the fully annotated AnnData object and produce condition-split pseudobulk aggregates for downstream differential expression or compositional analysis.

### 7.1 Export Refined Labels

In [ ]:
# saving h5ad
mmc_adata_output_proc_filepath = ( mapmycells_output_dir / 
                             "sn_mapmycells_integrated_processed_output.h5ad")
adata.write_h5ad(mmc_adata_output_proc_filepath)

# saving hvgs_res
hvgs_res_filepath = ( mapmycells_output_dir / 
                             "sn_hvgs_results.csv")
hvgs_res.to_csv(hvgs_res_filepath)

### 7.2 Export Condition-Split Pseudobulk Aggregates

We split cells by condition (`Control` vs. `PD`) and aggregate raw counts by refined cell type. These pseudobulk matrices are commonly used for differential expression analysis (e.g., DESeq2) or cell type composition comparisons.

In [ ]:
# assume adata.obs["condition_id"] has values like "control" and "PD"
adata_control = adata[adata.obs["condition_id"] == "Control"].copy()
adata_case    = adata[adata.obs["condition_id"] == "PD"].copy()

In [ ]:
# Pseudobulk aggregation: sum raw counts per refined cell type
control_sum = sc.get.aggregate(adata_control, by="cell_type_refined", func="sum", layer="counts")
case_sum    = sc.get.aggregate(adata_case,    by="cell_type_refined", func="sum", layer="counts")
all_sum     = sc.get.aggregate(adata,         by="cell_type_refined", func="sum", layer="counts")

In [ ]:
# Create output directories and save pseudobulk files
round1_output_dir = output_path / "refined_classes"
os.makedirs(round1_output_dir, exist_ok=True)

for folder in ["case", "control", "all"]:
    os.makedirs(round1_output_dir / folder, exist_ok=True)

control_sum.write_h5ad(round1_output_dir / "control" / "sum_counts.h5ad")
case_sum.write_h5ad(round1_output_dir / "case" / "sum_counts.h5ad")
all_sum.write_h5ad(round1_output_dir / "all" / "asap-crn_SN_subtyping_v1.0_sum_counts_by_class.h5ad")

## Files Produced by This Notebook

This notebook generates annotated outputs for downstream analysis. Exact filenames may vary by output path.

### Primary Outputs

- **Annotated AnnData (`.h5ad`)**  
  Final single-cell dataset with MapMyCells labels, confidence metrics, DA scores, and refined annotations.

- **Pseudobulk matrices**  
  Aggregated expression profiles by refined cell type and condition (for example, all samples, case, and control), suitable for DESeq2, edgeR, or other bulk RNA-seq workflows.

### Optional Outputs

- **Raw MapMyCells results (`.csv`)**  
  Per-cell mapping labels and confidence values.

- **Summary tables (`.csv`)**  
  Cell type counts or proportions.

- **Figures (`.png`, `.pdf`)**  
  UMAPs and QC plots generated during analysis.

### Common Added Metadata (`adata.obs`)

| Column | Description |
|---|---|
| `cell_type` | Broad cell type label |
| `prob` | Mapping confidence |
| `rho` | Correlation score |
| `DA_score` | Dopaminergic marker score |
| `cell_type_refined` | Final refined label |

## Summary and Next Steps

In this notebook, we used the processed Substantia Nigra single-cell dataset to perform reference-based cell type annotation with **MapMyCells**. Key steps included:

1. **Taxonomy mapping** to the Allen HMBA Basal Ganglia reference  
2. **Result standardization** with confidence metrics  
3. **Orthogonal validation** using dopaminergic marker scores  
4. **Label refinement** using multi-criteria thresholds  
5. **Export** of annotated data and condition-split pseudobulk outputs  

These annotations provide a reproducible foundation for downstream analyses such as cell type composition, differential expression, and cross-dataset comparisons.

### Explore These Outputs Further

- **Differential expression** using case/control pseudobulk outputs with DESeq2 or edgeR  
- **Cell type composition** comparisons between conditions  
- **Sub-population analysis** by reclustering specific cell types  
- **Reference comparison** using alternative taxonomies  
- **Marker discovery** across refined cell groups  

### Additional Resources

Explore more workflows and examples in the **ASAP-CRN Learning Lab** GitHub repository:

🔗 **https://github.com/ASAP-CRN/asap-crn-learning-lab**

This notebook is intended for use with approved data accessed through the ASAP-CRN Cloud.